# 09 — DQN Baseline

A value-based reinforcement learning baseline (Deep Q-Network) for comparison
against PPO. DQN uses a **discrete** action space (5 fixed VM-change choices),
in contrast to PPO's continuous control. This illustrates why continuous,
policy-gradient control is better suited to the multi-objective cost/SLA
tradeoff.

Uses the refactored modules: `env.py`, `agent.py` (DQN + train_dqn),
`evaluate.py` (run_dqn). No pasted code.

## 1. Imports

In [1]:
import json
import numpy as np
import torch

from env import CloudClusterEnv, STEPS_PER_WEEK
from agent import DQN, train_dqn, DISCRETE_ACTIONS
from evaluate import run_dqn, run_hpa, run_ppo

stats = json.load(open('trace_params.json'))['stats']
print("Imports ready. Discrete actions:", DISCRETE_ACTIONS)

Setup complete. Steps per week: 672
CloudClusterEnv defined.
Imports ready. Discrete actions: [-5, -2, 0, 2, 5]


## 2. Train the DQN baseline

Trains for 250k steps with epsilon-greedy exploration, a replay buffer, and a
target network (all defined in `agent.py`). Saves the model and its
convergence log for the training-curve figure.

**~15-20 min run.**

In [2]:
env = CloudClusterEnv(stats, seed=None)

print("Training DQN baseline (250k steps)...\n")
dqn_net, dqn_rewards, dqn_convergence = train_dqn(env, total_steps=250_000)

torch.save(dqn_net.state_dict(), 'dqn_baseline.pth')
json.dump(dqn_convergence, open('dqn_convergence.json', 'w'))
print("\nSaved dqn_baseline.pth and dqn_convergence.json")
if len(dqn_rewards) >= 2:
    print(f"First episode reward: {dqn_rewards[0]:.1f}")
    print(f"Last episode reward:  {dqn_rewards[-1]:.1f}")

Training DQN baseline (250k steps)...

steps  10000 | eps 0.81 | recent reward   -119.7
steps  20000 | eps 0.62 | recent reward    -62.8
steps  30000 | eps 0.43 | recent reward    -39.1
steps  40000 | eps 0.24 | recent reward    -25.8
steps  50000 | eps 0.05 | recent reward    -19.4
steps  60000 | eps 0.05 | recent reward    -16.1
steps  70000 | eps 0.05 | recent reward    -16.3
steps  80000 | eps 0.05 | recent reward    -18.2
steps  90000 | eps 0.05 | recent reward    -17.5
steps 100000 | eps 0.05 | recent reward    -16.0
steps 110000 | eps 0.05 | recent reward    -16.6
steps 120000 | eps 0.05 | recent reward    -22.5
steps 130000 | eps 0.05 | recent reward    -20.1
steps 140000 | eps 0.05 | recent reward    -25.1
steps 150000 | eps 0.05 | recent reward    -18.4
steps 160000 | eps 0.05 | recent reward    -16.4
steps 170000 | eps 0.05 | recent reward    -13.4
steps 180000 | eps 0.05 | recent reward    -16.3
steps 190000 | eps 0.05 | recent reward    -16.8
steps 200000 | eps 0.05 | rece

## 3. Evaluate DQN and build the three-way comparison

All three agents evaluated on the same fixed seeds for a fair, paired comparison.

In [3]:
# load the best PPO model for the comparison
from agent import ActorCritic
ppo_net = ActorCritic()
ppo_net.load_state_dict(torch.load('ppo_sla-focused.pth'))
ppo_net.eval()

# evaluate all three on the SAME fixed seeds (paired, consistent)
N = 5
seeds = [100 + i for i in range(N)]

def avg_over_seeds(agent_fn, needs_net=None):
    runs = []
    for s in seeds:
        env = CloudClusterEnv(stats, seed=s)
        if needs_net is not None:
            runs.append(agent_fn(env, needs_net))
        else:
            runs.append(agent_fn(env))
    return {k: float(np.mean([r[k] for r in runs])) for k in ['cost','breaches','util','vms']}

hpa = avg_over_seeds(run_hpa)
dqn = avg_over_seeds(run_dqn, dqn_net)
ppo = avg_over_seeds(run_ppo, ppo_net)

print("="*58)
print("THREE-WAY COMPARISON (same seeds, same objective)")
print("="*58)
print(f"{'METRIC':<16}{'HPA':>13}{'DQN':>13}{'PPO':>13}")
print("-"*58)
print(f"{'Total cost':<16}{hpa['cost']:>13.1f}{dqn['cost']:>13.1f}{ppo['cost']:>13.1f}")
print(f"{'SLA breaches':<16}{hpa['breaches']:>13.0f}{dqn['breaches']:>13.0f}{ppo['breaches']:>13.0f}")
print(f"{'Utilisation':<16}{hpa['util']:>13.3f}{dqn['util']:>13.3f}{ppo['util']:>13.3f}")
print(f"{'Avg VMs':<16}{hpa['vms']:>13.1f}{dqn['vms']:>13.1f}{ppo['vms']:>13.1f}")
print("="*58)

# save for the figures notebook
json.dump({'hpa': hpa, 'dqn': dqn, 'ppo': ppo},
          open('three_way_comparison.json', 'w'), indent=2)
print("\nSaved three_way_comparison.json")

THREE-WAY COMPARISON (same seeds, same objective)
METRIC                    HPA          DQN          PPO
----------------------------------------------------------
Total cost              291.0        169.7        177.5
SLA breaches             1694         3411         1037
Utilisation             0.560        0.970        0.950
Avg VMs                   8.7          5.1          5.3

Saved three_way_comparison.json
